# 39 Domain-Prior Router Evaluation

This keeps the trained domain-6 DQA-MoX checkpoint from 38, then evaluates a tiny client/domain router prior.  The intent is FedMoX-like deployment: shared learned MoE weights plus a small router adapter per client/domain, rather than six full detectors.

In [ ]:
from __future__ import annotations

import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "dynamic_quality_aware_classwise_aggregation").exists():
    REPO_ROOT = Path("/app/Object_Detection")

PROJECT_ROOT = REPO_ROOT / "dynamic_quality_aware_classwise_aggregation" / "scene_daynight_dqa"
RUNNER = PROJECT_ROOT / "aggressive_dqamox" / "scripts" / "run_39_domain_prior_router_eval_loop.py"
LOG_DIR = PROJECT_ROOT / "aggressive_dqamox" / "logs"
LOG_DIR.mkdir(parents=True, exist_ok=True)
LOG_PATH = LOG_DIR / f"39_domain_prior_router_eval_{datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')}.log"

cmd = [
    sys.executable,
    str(RUNNER),
    "--target-map50", "0.55",
    "--imgsz", "640",
    "--val-batch-size", "16",
]

print(" ".join(cmd))
print("log:", LOG_PATH)
with LOG_PATH.open("w", encoding="utf-8") as log:
    proc = subprocess.run(cmd, cwd=REPO_ROOT, stdout=log, stderr=subprocess.STDOUT)
print("returncode:", proc.returncode)
print(LOG_PATH.read_text(encoding="utf-8", errors="replace")[-8000:])
if proc.returncode not in (0, 2):
    raise SystemExit(proc.returncode)


In [ ]:
from __future__ import annotations

import csv
from pathlib import Path

summary_path = PROJECT_ROOT / "aggressive_dqamox" / "reports" / "39_domain_prior_router_eval_loop_summary.csv"
metrics_path = PROJECT_ROOT / "aggressive_dqamox" / "output" / "39_domain_prior_router_eval_loop" / "stats" / "39_domain_prior_router_metrics.csv"

if summary_path.exists():
    for row in csv.DictReader(summary_path.open(encoding="utf-8")):
        print("summary:", row.get("status"), row.get("best_label"), row.get("best_map50"), row.get("best_map50_95"))

if metrics_path.exists():
    rows = list(csv.DictReader(metrics_path.open(encoding="utf-8")))
    for row in rows:
        if row.get("split") == "domain_prior_total":
            print(row.get("checkpoint_label"), row.get("map50"), row.get("map50_95"), row.get("mode"), row.get("bias"))
